In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    base_url=os.environ.get("OPENAI_API_BASE")
)

# Multi-modal

## Image

In [ ]:
import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(city):
    image_response = client.images.generate(
        model="dall-e-3",
        prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
        n=1,
        response_format="b64_json"
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = artist("New York City")
display(image)

## Audio

注意:
- 请参考原教程，配置 ffmpeg

In [ ]:
from pydub import AudioSegment
from pydub.playback import play

def talker(message):
    response = client.audio.speech.create(
        model="tts-1",
        voice="alloy",
        input=message
    )

    audio_stream = BytesIO(response.content)
    audio = AudioSegment.from_file(audio_stream, format="mp3")
    play(audio)


In [ ]:
talker("神明神明请张开嘴，让我知道我是谁?")

## Agent Framework

In [ ]:
MODEL = 'gpt-4o-mini'

SYSTEM_PROMPT = (
    "You are a helpful assistant for an Airline called FlightAI. "
    "Give short, courteous answers, no more than 1 sentence. "
    "Always be accurate. If you don't know the answer, say so."
)

# --- Data ---
CITY_TICKET_PRICES = {
    "london": "$799",
    "paris": "$999",
    "tokyo": "$1099",
}

# --- Function Definitions ---

def get_flight_ticket_price(destination_city: str) -> str:
    """Retrieves the ticket price for a given city.

    Args:
        destination_city: The city to get the price for.

    Returns:
        The ticket price as a string, or "Price Not Found" if the city is not in the database.
    """
    city_lower = destination_city.lower()
    return CITY_TICKET_PRICES.get(city_lower, "Price Not Found")


get_price_tool_spec = {
    "name": "get_flight_ticket_price",
    "description": "Provides the price of a return flight ticket to the specified destination city. Use this whenever a user inquires about ticket prices.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the user wants to fly to.",
            },
        },
        "required": ["destination_city"],
    },
}

available_tools = [
    {"type": "function", "function": get_price_tool_spec},
]



def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    city = arguments.get('destination_city')
    price = get_flight_ticket_price(city)
    response = {
        "role": "tool",
        "content": json.dumps({"destination_city": city,"price": price}),
        "tool_call_id": tool_call.id
    }
    return response, city


def chat(history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=available_tools)
    image = None

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response, city = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        image = artist(city)
        response = client.chat.completions.create(model=MODEL, messages=messages)
    
    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]

    talker(reply)

    return history, image

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500)
    with gr.Row():
        entry = gr.Textbox(label="Chat with our AI Assistant:")
    with gr.Row():
        clear = gr.Button("Clear")
    
    def do_entry(message, history):
        history += [{"role": "user", "content": message}]
        return "", history
    
    entry.submit(do_entry, inputs=[entry, chatbot], outputs=[entry, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, image_output]
    )
    clear.click(lambda: None, inputs=None, outputs=chatbot, queue=False)

ui.launch(inbrowser=True)